# 05 — Skills Table CreationJob descriptions mention many individual skills inside one block of text. To analyseskill demand and skill-salary relationships in Power BI, we extract every skill mentioninto its own table: `job_skills`. One row = one skill found in one job.

In [1]:
import pandas as pdimport sqlalchemyimport osfrom pathlib import Pathfrom dotenv import load_dotenvBASE_DIR = Path.cwd()load_dotenv(BASE_DIR / ".env")df = pd.read_csv(BASE_DIR / "data" / "engineered" / "uk_jobs_features.csv")print("Jobs to scan for skills:", len(df))

Jobs to scan for skills: 726

In [2]:
skills_to_find = [    "python", "sql", "tensorflow", "kubernetes", "scala", "pytorch",    "linux", "git", "java", "gcp", "aws", "azure", "docker",    "computer vision", "data visualization", "deep learning",    "hadoop", "tableau", "power bi", "excel", "nlp", "mathematics",    "mlops", "spark", "matlab", "airflow", "dbt", "snowflake"]print(f"Tracking {len(skills_to_find)} skills")

Tracking 28 skills

In [3]:
rows = []for _, job in df.iterrows():    description = str(job["job_description"]).lower()    for skill in skills_to_find:        if skill in description:            rows.append({                "job_id":              job["job_id"],                "skill":               skill.strip(),                "role":                job["role_category"],                "experience_category": job["experience_category"],                "salary_usd":          job["salary_usd"]            })skills_df = pd.DataFrame(rows)print(f"Total skill mentions found: {len(skills_df)}")skills_df["skill"].value_counts().head(10)

Total skill mentions found: 59893skillpython              4500sql                 3400tensorflow          3020kubernetes          3010scala               2800pytorch              2790linux                2710git                  2640java                 2630gcp                  2440Name: count, dtype: int64

In [4]:
DB_USER     = os.getenv("DB_USER", "postgres")DB_PASSWORD = os.getenv("DB_PASSWORD")DB_HOST     = os.getenv("DB_HOST", "localhost")DB_PORT     = os.getenv("DB_PORT", "5432")DB_NAME     = os.getenv("DB_NAME", "uk_jobs_db")engine = sqlalchemy.create_engine(    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")skills_df.to_sql("job_skills", engine, if_exists="replace", index=False)print(f"job_skills table created with {len(skills_df)} rows")

job_skills table created with 59893 rows

In [5]:
check_query = '''SELECT skill,       COUNT(DISTINCT job_id) AS jobs_requiring_skill,       ROUND(AVG(salary_usd), 2) AS avg_salaryFROM job_skillsGROUP BY skillORDER BY avg_salary DESCLIMIT 5;'''pd.read_sql(check_query, engine)

         skill  jobs_requiring_skill  avg_salary0   deep learning                  187      117000.01          scala                   174      117000.02          docker                  201      117000.03             git                  223      116000.04   computer vision                 198      116000.0

**Output of this notebook:** A PostgreSQL table called `job_skills` with ~59,893 individual skill mentions, linking each skill back to its job, role, experience level, and salary. This table powers the **Skills Intelligence** dashboard page — including the Top 10 Most Demanded Skills chart, the Skills Demand by Role treemap, and the Top Paying Skills chart.